In [9]:
# 导入torchvision,用于处理图像和视频，转换成张量
from torchvision.transforms import ToTensor
import torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
# 加载数据集pytorch本身提供了经典的手写数据集
# 加载训练集
train_ds = torchvision.datasets.MNIST(
    'data/',
    # 指定加载训练集
    train=True,
    # 将图像，numpy转换为tensor
    transform=ToTensor(),
    download=True
)
# 加载测试训练集
test_ds = torchvision.datasets.MNIST(
    'data/',
    # 指定加载训练集
    train=False,
    # 将图像，numpy转换为tensor
    transform=ToTensor(),
    download=True
)
# 指定训练数据加载器
train_dl = torch.utils.data.DataLoader(
    train_ds,
    batch_size=64,
    # 提高模型泛化能力
    shuffle=True# 每轮epoch打乱样本顺序
)

test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size=256,
    # 提高模型泛化能力
    shuffle=True
)

class Model(nn.Module):
    # 初始化函数定义网络层
    def __init__(self):
        super().__init__()
        # 定义一个784个节点的线性层
        self.linear_1 = nn.Linear(28*28,120)
        self.linear_2 = nn.Linear(120,84)
        self.linear_3 = nn.Linear(84,10)

    # 定义前向传播函数
    def forward(self,input):
        # 将输入的照片转换为一个维度
        x = input.view(-1,784)
        # 将输入给到第一层线性层
        x = self.linear_1(x)
        # 使用relu激活函数
        x = F.relu(x)
        # 将输入给到第二层线性层
        x = self.linear_2(x)
        # 使用relu激活函数
        x = F.relu(x)
        # 将输入给到第三层线性层
        x = self.linear_3(x)
        # 使用relu激活函数
        x = F.relu(x)
        # 返回结果
        return x
        



In [10]:
model = Model()
# 加载好之前训练好的参数
modelParams = torch.load('mnistModel')
model.load_state_dict(modelParams)

C:\Users\rehuanoo\AppData\Local\Temp\ipykernel_13672\844304077.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modelParams = torch.load('mnistModel')


<All keys matched successfully>

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device
model = Model().to(device)

In [12]:
# 定义损失函数
loss_fn = torch.nn.CrossEntropyLoss()
# 权重衰减
# weight_decay=0.01
# 定义优化器
# optimizer = torch.optim.SGD(model.parameters(),lr=0.005,weight_decay=0.01)
optimizer = torch.optim.SGD(model.parameters(),lr=0.005)
# 定义训练函数，从训练加载器里提取训练数据，进行训练

In [ ]:
def train(dataLoader,model,loss_fn,optimizer):
    # 获取整个训练数据集大小
    size = len(dataLoader.dataset)
    train_loss,correct=0,0      
    # 循环遍历整个数据集
    for X,Y in dataLoader:
        X,Y=X.to(device),Y.to(device)
        # 进行初步预测
        pred=model(X)
        # 查看误差
        loss = loss_fn(pred,Y)

        # 梯度归零
        optimizer.zero_grad()
        # 反向传播
        loss.backward()
        # 使用优化器更新参数
        optimizer.step()

        # 计算当前准确率
        # 计算准确率不需要更新梯度
        with torch.no_grad():
            # 在10个数据里挑最大值
            correct += (pred.argmax(1)==Y).type(torch.float).sum().item()
            # 获取当前训练的损失值
            train_loss +=loss.item()
    # 计算平均损失和正确率
    train_loss = train_loss / size
    correct = correct/size
    return train_loss,correct
        
# 定义一个测试函数
def test(dataLoader,model):
    # 获取测试集大小
    size = len(dataLoader.dataset)
    train_loss, correct = 0, 0
     # 计算准确率不需要更新梯度
    with torch.no_grad():
        for X,Y in dataLoader:
            X,Y=X.to(device),Y.to(device)
            # 进行初步预测
            pred=model(X)
            # 查看误差
            loss = loss_fn(pred,Y)
            # 在10个数据里挑最大值
            correct += (pred.argmax(1)==Y).type(torch.float).sum().item()
            # 获取当前训练的损失值
            train_loss +=loss.item()
    # 计算平均损失和正确率
    train_loss = train_loss / size
    correct = correct/size
    return train_loss,correct
# 训练多轮
epochs = 15
train_acc=[]
test_acc=[]
for epoch in range(epochs):
    epoch_loss,epoch_acc = train(train_dl,model,loss_fn,optimizer)
    epoch_test_loss,epoch_test_acc = test(test_dl,model)
    train_acc.append(epoch_acc)
    test_acc.append(epoch_test_acc)
    print('epoch:{},train_loss:{},train_acc{}'.format(epoch,epoch_loss,epoch_acc))
    print('epoch:{},test_loss:{},test_acc{}'.format(epoch,epoch_test_loss,epoch_test_acc))
    print('---------------------------------')

epoch:0,train_loss:0.03539353709220886,train_acc0.31066666666666665
epoch:0,test_loss:0.008796438145637513,test_acc0.378
---------------------------------
epoch:1,train_loss:0.03179923031727473,train_acc0.37533333333333335
epoch:1,test_loss:0.007214653193950653,test_acc0.3994
---------------------------------
epoch:2,train_loss:0.02614394002755483,train_acc0.41701666666666665
epoch:2,test_loss:0.006192631411552429,test_acc0.4542
---------------------------------
epoch:3,train_loss:0.023786674386262894,train_acc0.4610166666666667
epoch:3,test_loss:0.005835578346252442,test_acc0.489
---------------------------------
epoch:4,train_loss:0.022805699257055917,train_acc0.4875
epoch:4,test_loss:0.005632072997093201,test_acc0.5089
---------------------------------
